# Study 808 — Continuing Overreaction — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_months': 184, 'spread_bps': 15.46, 't_nw': 0.58, 't_1s': 0.55, 'hi_bps': 152.94, 'lo_bps': 137.48, 'welch_t': 0.31, 'gross_sharpe': 0.14, 'placebo_obs': 15.46, 'placebo_mean': 1.33, 'placebo_sd': 20.82, 'placebo_p': 0.261, 'placebo_sigma': 0.68, 'placebo_draws': 1000, 'era_early_bps': 8.23, 'era_early_t': 0.26, 'era_early_n': 82, 'era_late_bps': 21.27, 'era_late_t': 0.54, 'era_late_n': 102, 'timer_1_gross': 15.46, 'timer_1_cost': 6.17, 'timer_1_net': 9.29, 'timer_1_t': 0.33, 'timer_5_gross': 15.46, 'timer_5_cost': 14.17, 'timer_5_net': 1.29, 'timer_5_t': 0.05, 'null_mean_t': -0.47, 'null_sd_t': 0.95, 'null_fire': 1, 'planted_t': 8.61, 'planted_welch': 5.86}

## The headline — long-high-CO / short-low-CO spread

Monthly equal-weight top-30% minus bottom-30% CO spread.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/month  NW(6) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : high-CO {R['hi_bps']:+.2f} vs low-CO {R['lo_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost, ann.)")

spread        : +15.46 bps/month  NW(6) t = +0.58  one-sample t = +0.55
books         : high-CO +152.94 vs low-CO +137.48 bps (Welch t = +0.31)
gross Sharpe  : 0.14 (before cost, ann.)


## Placebo — column-permute the forward returns (1,000 permutations)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.2f} "
      f"(sd {R['placebo_sd']:.2f}) -> p = {R['placebo_p']:.5f} ({R['placebo_sigma']:+.2f}sigma)")

observed +15.46 bps vs placebo mean +1.33 (sd 20.82) -> p = 0.26100 (+0.68sigma)


## Robustness — two eras (split 2018-01-01)

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=82): +8.23 bps  NW t = +0.26
2018-2026 (n=102): +21.27 bps  NW t = +0.54


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per monthly rebalance; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/month (cost {c:.2f}/reb, t={t:+.2f})")

 1 bp one-way: gross +15.46 -> net +9.29 bps/month (cost 6.17/reb, t=+0.33)
5 bps one-way: gross +15.46 -> net +1.29 bps/month (cost 14.17/reb, t=+0.05)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted continuation.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from continuing_overreaction import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=808+s, n_assets=40, n_days=1800))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.02, seed=808, n_assets=40, n_days=1800))
print(f"planted (edge=0.02): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean -0.27 (sd 0.95), |t|>=2 in 0/8


planted (edge=0.02): NW t = +8.61, Welch t = +5.86


## Verdict

- **Signal — None.** The Byun-Lim-Yun continuing-overreaction premium does **not** replicate on 50 liquid US mega-caps: the long-high-CO / short-low-CO spread is **+15.46 bps/month** (NW *t* = **+0.58**) — the right sign but statistically zero, ≈+0.68σ in a 1,000-permutation placebo (p = 0.261), flat in both eras (*t* = +0.26 / +0.54). The 20-seed synthetic control recovers a *planted* continuation cleanly (*t* = +8.61, fires on 1/20 nulls), so this is a genuine null, not machinery. Survivorship biases the magnitude upward if anything.
- **Tradability — Mirage.** The book is insignificant gross; the monthly round-trip friction eats it to **+9.29 bps/month** (*t* = +0.33) at 1 bp and **+1.29 bps** (*t* = +0.05) at 5 bps.